# Лабораторная работа: Бинарная классификация факта ДТП на основе данных о дорожном трафике

**Курс:** Машинное обучение и анализ данных  

---

## Источник данных

Датасет: **[Synthetic Traffic Benchmark](https://www.kaggle.com/datasets/razanihababdellatif/emergency-dispatch-and-traffic-physics-dataset)** (синтетический бенчмарк дорожного трафика)  
Платформа: [Kaggle](https://www.kaggle.com/)  

> Датасет содержит ~20 000 записей о дорожных событиях, включающих информацию о времени, типе дороги, погодных условиях, видимости, характеристиках водителя и транспортного средства, а также сведения о произошедших ДТП (факт аварии, тип столкновения, тяжесть, время реагирования и т.д.).

In [1]:
# Базовые библиотеки
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns

# Предобработка данных
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Модели
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# Метрики
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# Настройки отображения
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42

## 0. Постановка задачи

### Описание предметной области

В датасете собраны наблюдения за дорожным трафиком: для каждой записи известны контекстные условия (время суток, день недели, праздник), характеристики дороги (тип зоны, тип дороги, тип перекрёстка, ограничение скорости), погодные и визуальные условия (погода, состояние покрытия, видимость), параметры водителя и автомобиля (тип ТС, возраст водителя, стаж, возраст автомобиля), а также текущая скорость и индекс загруженности. Для части записей зафиксирован факт ДТП и его последствия.

### Формулировка задачи

**Тип задачи:** бинарная классификация.

**Целевая переменная:** `Accident_Occurred` ∈ {0, 1}
- `0` — ДТП не произошло;
- `1` — ДТП произошло.

**Признаки (предикторы):** все переменные, известные **в момент наблюдения до события**:
- временные: `Hour`, `Month`, `Day_of_Week`, `Time_of_Day`, `Is_Holiday`;
- дорожные: `Area_Type`, `Road_Type`, `Intersection_Type`, `Speed_Limit_kmh`;
- условия: `Weather`, `Road_Condition`, `Visibility_m`, `Congestion_Index`;
- водитель/ТС: `Vehicle_Type`, `Driver_Age`, `Driver_Experience_yrs`, `Vehicle_Age_yrs`;
- текущая динамика: `Recorded_Speed_kmh`.

**Исключаемые признаки (утечка данных / пост-фактум информация):**
`Collision_Type`, `Vehicles_Involved`, `Accident_Severity`, `Dispatch_Priority`, `Actual_Response_Time_min`, `Road_Closure_Duration_min`, `Baseline_Expected_Response_Time_min`, `Record_ID`, `Timestamp`, `Date`.

Причина: эти признаки становятся известны только *после* аварии и/или являются её прямыми следствиями — их использование привело бы к «протеканию» целевой переменной в признаки и завышенным, но бесполезным на практике метрикам.

### Прикладная цель

Построить модель, которая по условиям движения и характеристикам участника предсказывает вероятность ДТП. Это позволит:
- выявлять «опасные» комбинации условий (погода × видимость × скорость × тип дороги);
- приоритизировать профилактические меры и маршруты патрулирования;
- сравнить несколько классификаторов и подобрать оптимальную сложность модели (число соседей в kNN).

### Метрики качества

Из-за ожидаемой несбалансированности классов (ДТП — редкое событие) основной метрикой выбираем **F1-score** для положительного класса и **ROC-AUC**, вспомогательно — accuracy и матрицу рассогласования. Разбиение train/test — стратифицированное (`stratify=y`), чтобы сохранить долю классов.

### План работы

1. Загрузка и первичный осмотр данных.
2. Разбиение на обучающую и тестовую выборки.
3. Разведочный анализ (EDA): описательные статистики, распределения, корреляционная матрица, интерпретация.
4. Обработка выбросов.
5. Обработка пропущенных значений.
6. Кодирование категориальных признаков.
7. Нормализация числовых признаков (обязательна для kNN).
8. Обучение kNN-классификатора (обоснование выбора метода).
9. Оценка на train/test, матрица рассогласования, подбор гиперпараметра `k`.
10. Сравнение с другими классификаторами (LogisticRegression, RandomForest).
11. Борьба с несбалансированностью классов.
12. Отбор признаков / удаление коррелированных переменных.
13. Выводы.